# Model Context Protocol (MCP)

Model Context Protocol (MCP) is an open protocol that standardizes how applications provide tools and context to LLMs. LangChain agents can use tools defined on MCP servers using the langchain-mcp-adapters library.

>>>>>> Basically, <b>it is tool server</b>, but with standard 

>>>>>> think of old phone charger, there're various type

>>>>>> but <b>with MCP, it is similar to what USB-C did to all chargers. United it all</b>

It is a protocol (rule for tool to follow, if not follow it's just tool not considered as MCP tool):

4 rules summarized by Gemini 



<b>Rule 1: The "Self-Discovery" Rule (Naming & Metadata) (Discovery)</b>
In a normal script, you know what a function does because you wrote it. In MCP, the server must tell the AI what is available.

The Rule: Your function must have a Unique Name (no spaces, usually snake_case) and a Clear Description.

<b>Why?</b> The AI uses the description to decide when to use the tool. If your description is "Calculates thing," the AI will never call it. If it’s "Calculates the annual interest rate for a banking customer based on their credit score," the AI knows exactly when to trigger it.

==========================================================================

<b>Rule 2: The "Strict Input" Rule (JSON Schema) (Schema)</b>
You cannot just pass a "variable" like you do in Python. You must define a Schema.

The Rule: Every input variable must have a defined Type (string, integer, boolean, etc.) and a Description.

sender (AI) will send exactly this format

In [ ]:
{
    "jsonrpc": "2.0",
    "id": "request-123",
    "method": "tools/call",
    "params": {
        "name": "calculate_ltv",
        "arguments": {
        "argument1": 85000,
        "argument2": 100000
    }
  }
}

<b>jsonrpc</b>: Always "2.0". This tells your server which version of the "language" is being spoken.

<b>id</b>: A unique string or number. Your server must include this same ID in its reply so the AI knows which answer belongs to which question.

<b>method</b>: Always "tools/call" for tool execution.

<b>params.name</b>: The exact string name of your function (the one you put in the @mcp.tool() decorator).

<b>params.arguments</b>: This is the "payload." It is a dictionary where the keys match your function's argument names.


<b>Why?</b> This creates the "form" the AI fills out. The AI reads the schema and says, "Okay, I need to provide a customer_id which must be an integer."

==========================================================================

<b>Rule 3: The "Standard Envelope" Rule (JSON-RPC) (Protocol)</b>
This is the technical "algorithm" of how messages move.

The Rule: All communication must be wrapped in a JSON-RPC 2.0 envelope.

In [2]:
{
  "jsonrpc": "2.0",
  "id": "request-123",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "The calculated LTV is 85.00%"
      }
    ]
  }
}

{'jsonrpc': '2.0',
 'id': 'request-123',
 'result': {'content': [{'type': 'text',
    'text': 'The calculated LTV is 85.00%'}]}}

<b>Why?</b> It ensures that the "request" and "response" are never mixed up. Every message has an ID. When the server replies, it includes that same ID so the AI knows which question the answer belongs to.

==========================================================================

<b>Rule 4: The "Content Block" Rule (Output Format) (Output)</b>
A normal Python function might return a simple string or a list. An MCP tool must return a specific Content Object.

The Rule: You must wrap your result in a "content" array, usually as a text block: {"content": [{"type": "text", "text": "YOUR_RESULT_HERE"}]}.

<b>Why?</b> MCP supports more than just text. By following this rule, your tool could technically return images, file links, or even audio, and the AI will know how to handle each type.

pip install langchain-mcp-adapters

<b>langchain-mcp-adapters</b> enables agents to use tools defined across one or more MCP servers.

MultiServerMCPClient is <b>stateless by default</b>. 

1. Each tool invocation creates a fresh MCP ClientSession
2. executes the tool
3. then cleans up. 

See the stateful sessions section for more details.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient  ## Import here
from langchain.agents import create_agent


client = MultiServerMCPClient(  ## define
    {
        "math": {
            "transport": "stdio",  # Local subprocess communication
            "command": "python",
            # Absolute path to your math_server.py file
            "args": ["/path/to/math_server.py"],
        },
        "weather": {
            "transport": "http",  # HTTP-based remote server
            # Ensure you start your weather server on port 8000
            "url": "http://localhost:8000/mcp",
        }
    }
)

tools = await client.get_tools()  ## this is not common syntax as normally await need to be in async func
# but it can use due to langchain special RECL something
agent = create_agent( 
    "claude-sonnet-4-5-20250929",
    tools 
)
## ainvoke (asynchronous invoke) not just invoke 
math_response = await agent.ainvoke( 
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)
weather_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
)

# Custom server

In [ ]:
# pip install fastmcp <Use fastmcp Lib>

In [ ]:
from fastmcp import FastMCP

# Math 
mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")

# Weather
from fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

# Some Example from Gemini

## This is writing mcp tool with FastMCP

In [ ]:
from mcp.server.fastmcp import FastMCP

# 1. The "Server" (The USB Hub)
mcp = FastMCP("BankingTools")

# 2. The "Tool" (The USB Device)
@mcp.tool()
def calculate_ltv(loan_amount: float, property_value: float) -> str:
    """
    Calculate the Loan-to-Value (LTV) ratio for a mortgage application.
    Use this tool whenever a user asks about loan risks or equity.
    """
    # The 'Algorithm' behind the scenes (SDK) handles Rule 1 & 2: 
    # It converts this docstring and these types into a JSON Schema for the AI.
    
    if property_value <= 0:
        return "Error: Property value must be greater than zero."
    
    ltv = (loan_amount / property_value) * 100
    
    # Rule 4: The 'Algorithm' (SDK) automatically wraps this 
    # string into the required MCP "Content Block" format.
    return f"The calculated LTV is {ltv:.2f}%"

# No need for a print statement; the MCP server stays 'alive' 
# waiting for an AI to ask for it.

## This is to write without the mcp.tool decorator

In [ ]:
import sys
import json

def calculate_ltv(loan_amount, property_value):
    ltv = (loan_amount / property_value) * 100
    return f"{ltv:.2f}%"

# --- MANUALLY DOING WHAT THE DECORATOR DOES ---
def handle_mcp_requests():
    for line in sys.stdin: # Listen to the 'pipe'
        request = json.loads(line)
        
        # 1. Manual "Discovery" logic
        if request["method"] == "tools/list":
            response = {
                "jsonrpc": "2.0", "id": request["id"],
                "result": {"tools": [{
                    "name": "calculate_ltv",
                    "description": "Calculates loan-to-value ratio",
                    "inputSchema": { "type": "object", "properties": { ... } }
                }]}
            }
        
        # 2. Manual "Calling" logic
        elif request["method"] == "tools/call":
            args = request["params"]["arguments"]
            # Manual execution and wrapping
            data = calculate_ltv(args["loan_amount"], args["property_value"])
            response = {
                "jsonrpc": "2.0", "id": request["id"],
                "result": {"content": [{"type": "text", "text": data}]}
            }
        
        # 3. Manual Output
        sys.stdout.write(json.dumps(response) + "\n")

# This is the "Algorithm" the decorator hides from you.

# Transports

MCP supports different transport mechanisms for client-server communication.

## HTTP
The http transport (also referred to as streamable-http) uses HTTP requests for client-server communication. See the MCP HTTP transport specification for more details.